# 流体力学と音声合成：入門ノートブック

## このノートブックについて

`docs/notes/fluid-dynamics-and-sound-synthesis.md` でまとめた調査をもとに、
音の生成における流体力学的な基礎を体験的に探索します。

| セクション | テーマ | 見所 |
|---|---|---|
| 1 | Reynolds 数の声道解剖図 | どの音素でいつ乱流遷移が起きるか（可視化） |
| 2 | Lighthill 音響アナロジー | 音源種別と強度の速度スケーリング（可視化） |
| 3 | 声門乱流と声質 | 吸気ノイズによる 4 種類の声質合成（可聴） |
| 4 | Coanda 効果と非周期性 | ジェット偏向による jitter/shimmer モデル（可聴） |
| 5 | 格子ボルツマン法 (LBM) | D1Q3 LBM による音響管シミュレーション（可視化・可聴） |

### 既存ノートブックとの関係

| ノートブック | 扱う内容 |
|---|---|
| `frication-model-phase3.ipynb` | 摩擦音のハイブリッド合成（波動管 + フィルタ） |
| `glottal-source-models.ipynb` | 声帯モデル（LF モデル、Rosenberg） |
| `digital-waveguide-mesh-prototype.ipynb` | DWM 2D メッシュシミュレーション |
| **このノートブック** | **流体力学的な音源の基礎 + 吸気ノイズ合成 + LBM 入門** |

参照文書：`docs/notes/fluid-dynamics-and-sound-synthesis.md`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from IPython.display import Audio, display
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ダークテーマ（既存ノートブックと共通）
plt.rcParams.update({
    'figure.facecolor': '#1e293b', 'axes.facecolor': '#0f172a',
    'axes.edgecolor': '#475569',   'axes.labelcolor': '#cbd5e1',
    'xtick.color': '#94a3b8',      'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',       'grid.color': '#334155',
    'grid.linestyle': '--',        'grid.alpha': 0.6,
    'lines.linewidth': 1.6,
})

# 物理定数
FS       = 44100         # 出力サンプリングレート [Hz]
C        = 343.0         # 音速 [m/s]（20°C 乾燥空気）
RHO      = 1.2           # 空気密度 [kg/m³]
NU       = 1.5e-5        # 動粘性係数 [m²/s]
RE_CRIT  = 1800.0        # 乱流遷移の臨界 Reynolds 数（声道の実験値）
L_TUBE   = 0.17          # 声道長 [m]
N_SEC    = 44            # 声道モデルのセクション数
SEC_LEN  = L_TUBE / N_SEC  # 1セクションの長さ [m]

OUT_DIR = Path('../data/processed/analysis')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'空気の物性値:')
print(f'  密度      ρ = {RHO} kg/m³')
print(f'  動粘性係数 ν = {NU:.2e} m²/s')
print(f'  音速      c = {C} m/s')
print(f'  臨界 Re      = {RE_CRIT:.0f}')
print(f'セットアップ完了')

---
## 1. Reynolds 数の声道解剖図

**Reynolds 数**（Re）は流れが層流か乱流かを判定する無次元数です。

$$\text{Re} = \frac{U \cdot D_h}{\nu}$$

- $U$: 断面平均流速 [m/s]　　$D_h = 2\sqrt{A/\pi}$: 水力直径 [m]
- $\nu$: 動粘性係数 [m²/s]（空気: $1.5 \times 10^{-5}$）

声道における典型値：
- **母音 /a/**：声門体積速度 ≈ 200 cm³/s、広い声道 → Re ≈ 200〜2000 → **層流支配**
- **摩擦音 /s/**：収縮部が極めて小さい（≈ 0.1 cm²）→ 局所流速が急上昇 → Re >> Re_crit → **乱流遷移**

この乱流遷移こそが **摩擦音（fricatives）の音源** です。

In [ ]:
# ---- 声道断面積プロファイル [cm²]（44 セクション, グロッタル端→口唇端）----
def interp_areas(ctrl, n=N_SEC):
    """制御点から N セクションの断面積に補間する"""
    return np.interp(np.linspace(0,1,n), np.linspace(0,1,len(ctrl)), ctrl)

AREAS_A = interp_areas([1.1,1.1,5.3,2.0,1.1,1.5,3.1,5.3,7.1,9.1,
                        11.3,11.3,9.1,7.1,6.2,8.0,9.1,9.8,10.0,9.5,
                         8.8,7.9,7.2,6.8,6.5,6.3,6.2,6.1,6.0,6.1,
                         6.3,6.6,7.0,7.4,7.8,8.1,8.3,8.4,8.4,8.1,
                         7.7,7.1,6.5,6.0])

AREAS_I = interp_areas([1.1,1.1,8.0,8.0,8.0,8.0,8.0,8.0,4.5,2.0,
                        0.8,0.8,0.8,1.1,1.5,4.5,7.1,9.1,10.2,10.5,
                        10.5,10.2,9.8,9.5,9.1,8.7,8.4,8.1,7.8,7.5,
                         7.2,7.0,6.9,6.9,7.0,7.2,7.5,7.8,8.0,8.1,
                         8.0,7.8,7.5,7.1])

# /s/ は前方歯列間に強い収縮（I形状をベースに）
AREAS_S = AREAS_I.copy()
AREAS_S[35:42] = 0.12   # 歯列間の収縮部 [cm²]

# /h/ は声門付近に弱い収縮（声門吸気音）
AREAS_H = AREAS_A.copy()
AREAS_H[0:5] = 0.5      # 声門〜咽頭の収縮


def reynolds_profile(areas_cm2, Q_ccs=200.0):
    """声道各セクションの Reynolds 数プロファイルを計算する。
    areas_cm2: 断面積配列 [cm²]
    Q_ccs    : 体積速度 [cm³/s]
    """
    A_m2     = areas_cm2 * 1e-4          # cm² → m²
    Q_m3ps   = Q_ccs * 1e-6              # cm³/s → m³/s
    U        = Q_m3ps / A_m2             # 断面平均流速 [m/s]
    D_h      = 2.0 * np.sqrt(A_m2 / np.pi)  # 水力直径 [m]（円形断面近似）
    return U * D_h / NU


x_sec  = np.arange(N_SEC) * SEC_LEN * 100   # 位置軸 [cm]
re_a   = reynolds_profile(AREAS_A, Q_ccs=200)
re_i   = reynolds_profile(AREAS_I, Q_ccs=200)
re_s   = reynolds_profile(AREAS_S, Q_ccs=200)
re_h   = reynolds_profile(AREAS_H, Q_ccs=300)  # /h/ はやや多い声門流量

print(f'/a/ 最大 Re = {re_a.max():.0f}  @ sec {re_a.argmax()}')
print(f'/i/ 最大 Re = {re_i.max():.0f}  @ sec {re_i.argmax()}')
print(f'/s/ 最大 Re = {re_s.max():.0f}  @ sec {re_s.argmax()}  '
      f'→ {"乱流" if re_s.max() > RE_CRIT else "層流"}')
print(f'/h/ 最大 Re = {re_h.max():.0f}  @ sec {re_h.argmax()}  '
      f'（声門付近）')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

datasets = [
    ('/a/ （母音）',  re_a, AREAS_A, '#34d399'),
    ('/i/ （母音）',  re_i, AREAS_I, '#60a5fa'),
    ('/s/ （摩擦音）', re_s, AREAS_S, '#f59e0b'),
    ('/h/ （声門吸気）', re_h, AREAS_H, '#a78bfa'),
]

for ax, (label, re, areas, col) in zip(axes, datasets):
    turb = re > RE_CRIT
    # Reynolds 数プロファイル
    ax2 = ax.twinx()
    ax2.fill_between(x_sec, areas, alpha=0.12, color=col)
    ax2.plot(x_sec, areas, color=col, linewidth=1.2, linestyle=':', alpha=0.6)
    ax2.set_ylabel('断面積 [cm²]', color='#64748b', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#64748b', labelsize=7)

    ax.fill_between(x_sec, re, where=turb,
                    alpha=0.45, color='#ef4444', label='乱流域 (Re > Re_crit)')
    ax.fill_between(x_sec, re, where=~turb, alpha=0.25, color=col)
    ax.plot(x_sec, re, color=col, linewidth=2.0, label=label)
    ax.axhline(RE_CRIT, color='#ef4444', linestyle='--',
               linewidth=1.5, label=f'Re_crit = {RE_CRIT:.0f}')
    ax.set_xlabel('声道位置（グロッタル端→口唇端）[cm]')
    ax.set_ylabel('Reynolds 数')
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True)
    ax.set_xlim(0, L_TUBE * 100)

fig.suptitle('Reynolds 数の声道解剖図：母音は層流、/s/ は乱流へ遷移する', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fd_re_vocal_tract_map.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()
print('Re 解剖図を保存しました')

---
## 2. Lighthill 音響アナロジー：音源種別と強度スケーリング

Lighthill（1952）は Navier-Stokes 方程式を変形して、流体中の音源を
**単極（Monopole）**・**双極（Dipole）**・**四重極（Quadrupole）** の 3 種類に分類しました。

| 音源種別 | 発生機構 | 強度スケーリング | 声道での例 |
|---|---|---|---|
| 単極 | 体積変化（声門開閉） | $P \propto U^4$ | 声帯の開閉パルス |
| 双極 | 固体への非定常力 | $P \propto U^6$ | 乱流ジェットが歯に衝突 |
| 四重極 | 乱流応力（Reynolds 応力） | $P \propto U^8$ | 声門後方の乱流 |

低マッハ数（$M = U/c \ll 1$）の発声条件では、**双極が主要な音源** です（単極は声門体積速度を指定すれば自動的に含まれる）。

**実用的な意味**：流速が2倍になると
- 単極は $2^4 = 16$ 倍
- 双極は $2^6 = 64$ 倍
- 四重極は $2^8 = 256$ 倍

摩擦音は**双極・四重極が支配的**なため、息の強さ（流速）に非常に敏感です。

In [ ]:
U_vec = np.linspace(0.05, 3.0, 400)   # 流速範囲 [m/s]
U_ref = 0.5                            # 参照流速 [m/s]

# 正規化した音源強度（U_ref のとき = 1）
P_mono = (U_vec / U_ref) ** 4
P_dipo = (U_vec / U_ref) ** 6
P_quad = (U_vec / U_ref) ** 8

# 実際の声道内流速（典型値）
U_vowel = 0.15   # 母音 /a/ での声道内流速 [m/s]
U_fric  = 1.8    # /s/ での収縮部流速 [m/s]
U_glot  = 0.8    # 声門での流速 [m/s]（ブレシー声）

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ---- 左：対数スケールで強度 vs 流速 ----
ax1.semilogy(U_vec, P_mono, color='#34d399', lw=2.5, label='単極 ∝ U⁴（声門開口）')
ax1.semilogy(U_vec, P_dipo, color='#f59e0b', lw=2.5, label='双極 ∝ U⁶（障害物衝突）')
ax1.semilogy(U_vec, P_quad, color='#ef4444', lw=2.5, label='四重極 ∝ U⁸（乱流応力）')

# 実際の流速帯を縦線で示す
ax1.axvline(U_vowel, color='#60a5fa', ls=':', lw=1.8, label=f'母音帯 U≈{U_vowel}m/s')
ax1.axvline(U_glot,  color='#a78bfa', ls=':', lw=1.8, label=f'声門帯 U≈{U_glot}m/s')
ax1.axvline(U_fric,  color='#fbbf24', ls=':', lw=1.8, label=f'摩擦音帯 U≈{U_fric}m/s')

ax1.set_xlabel('流速 U [m/s]', fontsize=11)
ax1.set_ylabel('音源強度（相対値）', fontsize=11)
ax1.set_title('Lighthill 音響アナロジー：音源種別と速度依存性', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True)
ax1.set_xlim(0.05, 3.0)

# ---- 右：流速 2 倍時の増幅倍率 ----
names   = ['単極\n(U⁴)', '双極\n(U⁶)', '四重極\n(U⁸)']
factors = [2**4, 2**6, 2**8]
colors  = ['#34d399', '#f59e0b', '#ef4444']
bars    = ax2.bar(names, factors, color=colors, alpha=0.85,
                   edgecolor='#475569', width=0.6)
for bar, val in zip(bars, factors):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 3,
             f'{val}×', ha='center', va='bottom',
             fontsize=15, color='#e2e8f0', fontweight='bold')
ax2.set_ylabel('流速 2 倍時の強度増加倍率', fontsize=11)
ax2.set_title('なぜ摩擦音は息の強さに敏感か', fontsize=11)
ax2.grid(True, axis='y')
ax2.set_yscale('log')

# 摩擦音 /s/ の双極強度 vs 母音帯の比較
ratio_dipo = (U_fric/U_ref)**6 / (U_vowel/U_ref)**6
print(f'摩擦音 /s/ (U={U_fric}m/s) の双極強度は')
print(f'  母音 (U={U_vowel}m/s) の双極強度より {ratio_dipo:.0f} 倍大きい')
print(f'  同じ比での単極差：{(U_fric/U_vowel)**4:.0f} 倍')

fig.suptitle('音源種別の速度スケーリング（Lighthill 1952）', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fd_lighthill_source_scaling.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()

---
## 3. 声門乱流と声質：吸気ノイズ（Aspiration Noise）の合成

声帯が完全に閉じない発声や気息性の強い発声では、声門付近で乱流が発生します。
この**声門乱流（aspiration noise /声門吸気ノイズ）**が声質を決める重要な成分です。

### 声質と声門パラメータの関係

$$\text{ノイズ振幅} \propto \text{OQ}^2 \quad (\text{OQ} = \text{開口率})$$

| 声質 | 開口率 OQ | 吸気ノイズ | 特徴 |
|---|---|---|---|
| 緊迫声（pressed） | 0.25 | 極少 | 声帯が強く閉じる、高圧 |
| 常態発声（modal） | 0.50 | 少 | 標準的な発声 |
| 気息声（breathy） | 0.72 | 多 | 弱い閉鎖、息漏れ |
| 囁き（whisper） | 0.90 | 主体 | 声帯振動なし |

合成パイプライン：
```
半正弦波グロッタルパルス (有声)    ─┐
                                   ├─ 加算 ─→ フォルマントフィルタ ─→ 出力
ホワイトノイズ + LP フィルタ (吸気) ─┘
```

In [ ]:
def make_glottal_source(n_samples, f0=140.0, oq=0.5,
                         jitter_frac=0.003, seed=42, fs=FS):
    """半正弦波グロッタルパルス列を生成する。
    oq: open quotient（開口率）= 開口時間 / 基本周期
    """
    T0_base  = int(fs / f0)
    T_open   = int(oq * T0_base)
    rng      = np.random.default_rng(seed)
    output   = np.zeros(n_samples)
    pos      = 0
    while pos < n_samples - T0_base:
        # 微小なジッターを加算して自然な声にする
        T0 = int(T0_base * (1.0 + rng.uniform(-jitter_frac, jitter_frac)))
        T0 = max(int(fs/400), min(int(fs/60), T0))
        T_op = int(oq * T0)
        pulse = np.sin(np.pi * np.arange(T_op) / T_op)   # 半正弦波
        end   = min(pos + T_op, n_samples)
        output[pos:end] += pulse[:end-pos]
        pos += T0
    return output


def make_aspiration_noise(n_samples, level=0.1,
                           cutoff_hz=3000.0, seed=99, fs=FS):
    """声門乱流ノイズ（吸気ノイズ）を生成する。
    声門形状による高域制限（カットオフ ~3 kHz）を模擬する。
    """
    rng   = np.random.default_rng(seed)
    noise = rng.standard_normal(n_samples)
    b, a  = signal.butter(3, cutoff_hz / (fs / 2.0), btype='low')
    noise = signal.lfilter(b, a, noise)
    noise *= level
    return noise


def apply_vowel_formants(source, vowel='a', fs=FS):
    """日本語5母音のフォルマントフィルタ（2次バンドパス直列接続）
    フォルマント周波数・帯域幅は平均的な成人男性の値を使用。
    """
    formants = {
        'a': [(850, 130),  (1610, 200), (2755, 300)],
        'i': [(270,  60),  (2290, 200), (3010, 250)],
        'u': [(300,  80),  ( 870, 120), (2240, 200)],
        'e': [(530,  90),  (1840, 180), (2480, 220)],
        'o': [(500,  90),  (1070, 140), (2340, 200)],
    }
    out = source.copy()
    nyq = fs / 2.0
    for f_n, bw in formants[vowel]:
        lo  = max(50, f_n - bw * 2.5) / nyq
        hi  = min(0.998, (f_n + bw * 2.5) / nyq)
        sos = signal.butter(2, [lo, hi], btype='bandpass', output='sos')
        out = signal.sosfilt(sos, out)
    pk = np.max(np.abs(out))
    return out / pk * 0.80 if pk > 1e-10 else out


def synthesize_voice_type(oq, asp_level, voiced=True,
                           f0=140.0, vowel='a',
                           duration=0.6, fs=FS):
    """声質パラメータから母音を合成する。"""
    N    = int(duration * fs)
    fade = int(0.015 * fs)
    env  = np.ones(N)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)

    glot = make_glottal_source(N, f0=f0, oq=oq) if voiced else np.zeros(N)
    asp  = make_aspiration_noise(N, level=asp_level)

    # 有声 + 吸気ノイズを合成してフォルマントフィルタへ
    pk_g = np.max(np.abs(glot))
    if pk_g > 1e-10:
        glot = glot / pk_g * (1.0 - asp_level)
    source = glot + asp

    wav = apply_vowel_formants(source, vowel=vowel, fs=fs)
    wav = (wav * env).astype(np.float32)
    return np.clip(wav, -0.95, 0.95)


print('音源・フィルタ関数 定義完了')

In [ ]:
voice_types = [
    {'name': '緊迫声\n(pressed)', 'f0': 200, 'oq': 0.25,
     'asp': 0.008, 'voiced': True,  'color': '#ef4444'},
    {'name': '常態発声\n(modal)',  'f0': 140, 'oq': 0.50,
     'asp': 0.045, 'voiced': True,  'color': '#34d399'},
    {'name': '気息声\n(breathy)', 'f0': 110, 'oq': 0.72,
     'asp': 0.32,  'voiced': True,  'color': '#60a5fa'},
    {'name': '囁き\n(whisper)',   'f0': 120, 'oq': 0.90,
     'asp': 0.82,  'voiced': False, 'color': '#a78bfa'},
]

print('各声質を合成中...')
wavs = []
for vt in voice_types:
    wav = synthesize_voice_type(
        oq=vt['oq'], asp_level=vt['asp'],
        voiced=vt['voiced'], f0=vt['f0'])
    wavs.append(wav)
    print(f'  {vt["name"].replace(chr(10), " ")}: OQ={vt["oq"]:.2f}  '
          f'吸気ノイズ={vt["asp"]:.3f}')

# ---- 可視化 ----
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.35)

T_SHOW = int(0.06 * FS)   # 先頭 60 ms を表示

for ci, (vt, wav) in enumerate(zip(voice_types, wavs)):
    t_ms = np.arange(T_SHOW) / FS * 1000

    # 波形
    ax_w = fig.add_subplot(gs[0, ci])
    ax_w.plot(t_ms, wav[:T_SHOW], color=vt['color'], linewidth=0.9)
    ax_w.set_ylim(-1, 1)
    ax_w.set_title(f"{vt['name']}\nOQ={vt['oq']:.2f}  Asp={vt['asp']:.3f}",
                   fontsize=8.5)
    ax_w.set_xlabel('時間 [ms]')
    ax_w.set_ylabel('振幅')
    ax_w.grid(True)

    # パワースペクトル密度
    ax_s = fig.add_subplot(gs[1, ci])
    f_w, pxx = signal.welch(wav, fs=FS, nperseg=2048)
    ax_s.plot(f_w, 10*np.log10(pxx+1e-12), color=vt['color'], linewidth=1.2)
    ax_s.set_xlim(0, 5000)
    ax_s.set_xlabel('周波数 [Hz]')
    if ci == 0:
        ax_s.set_ylabel('PSD [dB/Hz]')
    ax_s.grid(True)

fig.suptitle('声門パラメータと声質：4 種類の /a/ 比較\n'
             '（上：波形  下：パワースペクトル密度）', fontsize=11)
plt.savefig(OUT_DIR / 'fd_voice_quality_comparison.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()

# ---- 試聴 ----
print('\n▼ 試聴（各声質のの違いを確認してください）')
for vt, wav in zip(voice_types, wavs):
    label = vt['name'].replace('\n', ' ')
    print(f"\n▶ {label}  (f0={vt['f0']} Hz, OQ={vt['oq']:.2f}, "
          f"吸気ノイズ={vt['asp']:.3f})")
    display(Audio(wav, rate=FS))

---
## 4. Coanda 効果と声の非周期性（Jitter / Shimmer）

**Coanda 効果**とは、気流が壁面に付着しやすい性質です。声門から放出されたジェットは、
声道の左右どちらかの壁に偏向することがあります。

- **実験的証拠**：Neubauer ら（JASA 2007）の静的声帯模型実験で PIV 計測により確認
- **音声への影響**：ジェットが偏向すると有効な声門断面積が変化し、体積速度が変動する
  → 声の基本周波数（ジッター）と振幅（シマー）の不規則性が生まれる

### 確率的モデル

```
各サイクルで確率 p で偏向を判定
偏向時：有効声門面積 × (0.75〜0.95) → 体積速度低下 → 振幅・周期が微妙に変わる
```

この確率的変動が、完全に周期的なモデルには出せない**声の自然な「生きた」感**を生み出します。

In [ ]:
def synth_coanda(n_samples, f0=140.0, oq=0.5, asp_level=0.05,
                  p_coanda=0.0, vowel='a', seed=7, fs=FS):
    """Coanda 効果の確率的モデルを含むグロッタル音源を合成する。
    p_coanda: ジェット偏向が起きる確率 (0.0=なし → 0.6=強い偏向)
    """
    T0_base  = int(fs / f0)
    T_open   = int(oq * T0_base)
    rng      = np.random.default_rng(seed)
    source   = np.zeros(n_samples)
    pos      = 0
    periods  = []
    amps     = []

    while pos < n_samples - int(T0_base * 1.5):
        # 微小ジッター（自然な揺らぎ）
        T0 = int(T0_base * (1.0 + 0.003 * rng.standard_normal()))
        T0 = max(int(fs/400), min(int(fs/60), T0))

        # Coanda 偏向判定
        if rng.random() < p_coanda:
            # 偏向時：有効断面積が縮小 → 体積速度低下
            area_factor = rng.uniform(0.72, 0.93)
            # 偏向はわずかに周期を延長することがある
            T0 = int(T0 * (1.0 + 0.025 * rng.random()))
        else:
            area_factor = 1.0

        T_op  = int(oq * T0)
        amp   = area_factor               # 体積速度 ∝ 有効面積（Bernoulli 近似）
        pulse = amp * np.sin(np.pi * np.arange(T_op) / T_op)
        end   = min(pos + T_op, n_samples)
        source[pos:end] += pulse[:end-pos]

        periods.append(T0)
        amps.append(amp)
        pos += T0

    # 吸気ノイズを加算
    source += make_aspiration_noise(n_samples, level=asp_level, seed=seed+1, fs=fs)

    # 声道フィルタ
    wav  = apply_vowel_formants(source, vowel=vowel, fs=fs)
    fade = int(0.015 * fs)
    env  = np.ones(n_samples)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    wav  = (wav * env).astype(np.float32)

    periods = np.array(periods)
    amps    = np.array(amps)
    jitter_pct  = np.std(periods)  / np.mean(periods)  * 100
    shimmer_pct = np.std(amps)     / np.mean(amps)     * 100

    return np.clip(wav, -0.95, 0.95), periods, amps, jitter_pct, shimmer_pct


N_C = int(0.7 * FS)
coanda_cases = [
    ('p=0.0\n完全周期',   0.0,  '#94a3b8'),
    ('p=0.2\n弱い偏向', 0.2,  '#34d399'),
    ('p=0.4\n中程度',   0.4,  '#f59e0b'),
    ('p=0.6\n強い偏向', 0.6,  '#ef4444'),
]

results_c = []
print('Coanda モデルを合成中...')
for label, p, col in coanda_cases:
    wav, periods, amps, j_pct, s_pct = synth_coanda(N_C, f0=140, p_coanda=p)
    results_c.append((label, p, col, wav, periods, amps, j_pct, s_pct))
    print(f'  {label.replace(chr(10), " ")}: '
          f'jitter={j_pct:.2f}%  shimmer={s_pct:.2f}%')

# ---- 可視化 ----
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

T_SHOW = int(0.12 * FS)
for ci, (label, p, col, wav, periods, amps, j_pct, s_pct) in enumerate(results_c):
    t_ms = np.arange(T_SHOW) / FS * 1000

    axes[0, ci].plot(t_ms, wav[:T_SHOW], color=col, linewidth=0.8)
    axes[0, ci].set_title(
        f'{label}\njitter={j_pct:.2f}%  shimmer={s_pct:.2f}%', fontsize=8.5)
    axes[0, ci].set_ylim(-1, 1)
    axes[0, ci].set_xlabel('時間 [ms]')
    axes[0, ci].grid(True)

    axes[1, ci].plot(periods, 'o-', color=col, markersize=3.5, linewidth=1.0)
    axes[1, ci].axhline(np.mean(periods), color='#94a3b8',
                        linewidth=1.0, linestyle='--', label='平均')
    axes[1, ci].set_xlabel('サイクル番号')
    axes[1, ci].set_ylabel('周期 [samples]')
    axes[1, ci].set_title('周期変動（ジッター）', fontsize=9)
    axes[1, ci].grid(True)

fig.suptitle('Coanda 効果の確率的モデル：ジェット偏向 → 声の非周期性', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fd_coanda_jitter_model.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()

print('\n▼ 試聴（機械的な声 → 自然な声への変化を確認してください）')
for label, p, col, wav, *_ in results_c:
    print(f'\n▶ {label.replace(chr(10), " ")}')
    display(Audio(wav, rate=FS))

---
## 5. 格子ボルツマン法（LBM）入門：D1Q3 による音響管シミュレーション

### LBM とは

格子ボルツマン法（Lattice Boltzmann Method）は、Navier-Stokes 方程式を直接解かず、
**ミクロな「粒子分布関数」の衝突と流れ**を格子上でシミュレートします。

```
各格子点 x での分布関数: f_q(x, t)  (q = 左向き, 静止, 右向き)

1. 衝突（BGK 近似）: f_q → f_q - (f_q - f_q^eq) / τ
2. ストリーミング:   f_q(x, t+dt) = f_q*(x - e_q·dt, t)
```

### DWM との比較

| 項目 | デジタル波動管（DWM） | 格子ボルツマン（LBM D1Q3） |
|---|---|---|
| 基礎方程式 | 1D 波動方程式 | Navier-Stokes（音響近似） |
| 変数 | 右進行波・左進行波 | 3方向の粒子分布関数 |
| 音速の扱い | 直接指定 | cs = 1/√3（格子単位）から決まる |
| 粘性損失 | 反射係数で近似 | 緩和時間 τ で制御 |
| 複雑形状 | 反射係数の変更が必要 | 格子で自然に表現 |
| 将来拡張 | 2D/3D DWM | 2D/3D + 流れ（乱流）の統一計算 |

### D1Q3 のパラメータ設定

物理音速 $c_{\rm phys}$ を再現するには：

$$\Delta t = \frac{\Delta x}{c_{\rm phys} \cdot \sqrt{3}}$$

これにより格子単位の音速 $c_s = 1/\sqrt{3}$ が物理音速に対応します。

In [ ]:
class LBMAcousticTube1D:
    """D1Q3 格子ボルツマン法による 1 次元音響管シミュレーション。

    格子速度  : e = [-1, 0, +1]（格子単位 lu/ts）
    重み      : w = [1/6, 2/3, 1/6]
    格子音速  : cs = 1/√3 lu/ts
    物理音速  : cs_phys = cs × (dx/dt) = c（dx と dt を適切に選ぶ）
    """

    def __init__(self, N, L_m, c_phys=343.0, tau=0.52):
        self.N      = N
        self.c_phys = c_phys
        self.dx     = L_m / N                          # 格子間隔 [m]
        self.dt     = self.dx / (c_phys * np.sqrt(3))  # 時間ステップ [s]
        self.fs_lbm = 1.0 / self.dt                    # LBM サンプリングレート [Hz]
        self.tau    = tau
        self.cs2    = 1.0 / 3.0   # 音速の 2 乗（格子単位）
        self.w      = np.array([1/6, 2/3, 1/6])
        self.e      = np.array([-1,  0,  1])
        self.reset()

    def reset(self):
        """静止状態（ρ=1, u=0）で初期化する"""
        self.f = np.outer(np.ones(self.N), self.w)  # shape: (N, 3)

    def get_density(self):
        return self.f.sum(axis=1)

    def step(self, source_pos=None, source_amp=0.0):
        """1 ステップ進める。口唇端（x=N-1）の圧力変動を返す。"""
        f = self.f

        # ---- マクロ量の計算 ----
        rho = f.sum(axis=1)                  # 密度
        u   = (f[:, 2] - f[:, 0]) / rho     # 速度

        # ---- 平衡分布 f_eq（2 次 BGK 展開）----
        f_eq = np.empty_like(f)
        for q in range(3):
            eu          = self.e[q] * u
            f_eq[:, q]  = self.w[q] * rho * (
                1.0
                + eu / self.cs2
                + 0.5 * eu**2 / self.cs2**2
                - 0.5 * u**2  / self.cs2
            )

        # ---- BGK 衝突 ----
        f_post = f - (f - f_eq) / self.tau

        # ---- 単極音源注入（全方向に均等に圧力変動を加える）----
        if source_pos is not None:
            f_post[source_pos] += self.w * source_amp

        # ---- ストリーミング（roll で周期 BC を使い、後で境界を上書き）----
        f_new = np.empty_like(f_post)
        f_new[:, 0] = np.roll(f_post[:, 0], -1)  # 左向き: x ← x+1
        f_new[:, 1] = f_post[:, 1]                # 静止  : 移動なし
        f_new[:, 2] = np.roll(f_post[:, 2], +1)  # 右向き: x ← x-1

        # ---- 境界条件 ----
        # 左端 (x=0) : 剛体壁（完全反射バウンスバック）
        f_new[0, 2] = f_post[0, 0]

        # 右端 (x=N-1) : 開放端（圧力解放: ρ = ρ₀ = 1.0）
        f_new[-1, 0] = 1.0 - f_new[-1, 1] - f_new[-1, 2]

        self.f = f_new
        return float(rho[-1] - 1.0)   # 口唇端の圧力変動を返す

    def run_impulse(self, duration_s):
        """インパルス応答（時系列）を計算する"""
        n_steps = int(duration_s * self.fs_lbm)
        output  = np.zeros(n_steps)
        for n in range(n_steps):
            amp      = 0.01 if n == 0 else 0.0
            output[n] = self.step(source_pos=0, source_amp=amp)
        return output


# ---- LBM の初期化 ----
N_LBM   = 44       # 格子数（DWM のセクション数と対応）
TAU_LBM = 0.52     # 緩和時間 τ > 0.5 で安定
lbm     = LBMAcousticTube1D(N_LBM, L_TUBE, tau=TAU_LBM)

print(f'LBM パラメータ:')
print(f'  格子数          N   = {lbm.N}')
print(f'  格子間隔        dx  = {lbm.dx*1000:.2f} mm')
print(f'  時間ステップ    dt  = {lbm.dt*1e6:.3f} μs')
print(f'  サンプリングレート  = {lbm.fs_lbm/1e3:.1f} kHz')
print(f'  緩和時間        τ   = {lbm.tau}')
print()
print(f'理論共鳴周波数（閉端-開放端管 L={L_TUBE*100:.0f}cm）:')
for k in range(1, 6):
    fn = (2*k-1) * C / (4 * L_TUBE)
    print(f'  f{k} = {fn:.0f} Hz', end='  ')
print()

In [ ]:
# ---- 波動伝播のスナップショット（インパルス入力後の圧力場）----

# 入力インパルスが管を往復するのに必要なステップ数を確認
# 片道: L/c の物理時間 = L/(c*dt) ステップ
steps_one_way = L_TUBE / (C * lbm.dt)
print(f'インパルスが管を片道する時間: {L_TUBE/C*1e6:.1f} μs = {steps_one_way:.0f} ステップ')

snap_steps = [0, 8, 16, 32, 48, 96]
snap_data  = {}
for target in snap_steps:
    lbm.reset()
    for n in range(target + 1):
        amp = 0.01 if n == 0 else 0.0
        lbm.step(source_pos=0, source_amp=amp)
    snap_data[target] = lbm.get_density() - 1.0

x_nodes = np.arange(N_LBM) * lbm.dx * 100   # [cm]
colors_snap = ['#34d399', '#60a5fa', '#f59e0b', '#ef4444', '#a78bfa', '#f472b6']

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for i, (step, col) in enumerate(zip(snap_steps, colors_snap)):
    t_us = step * lbm.dt * 1e6
    axes[i].fill_between(x_nodes, snap_data[step], alpha=0.25, color=col)
    axes[i].plot(x_nodes, snap_data[step], color=col, linewidth=2.0)
    axes[i].axhline(0, color='#475569', linewidth=0.8)
    axes[i].set_title(f't = {t_us:.1f} μs  (step={step})', fontsize=9)
    axes[i].set_xlabel('位置（グロッタル端→口唇端） [cm]')
    axes[i].set_ylabel('圧力変動 ρ\'')
    axes[i].set_xlim(0, L_TUBE * 100)
    # 進行方向を矢印で示す
    if step <= int(steps_one_way):
        axes[i].set_title(f't = {t_us:.1f} μs  →右進行', fontsize=9)
    else:
        axes[i].set_title(f't = {t_us:.1f} μs  ←左進行（反射後）', fontsize=9)
    axes[i].grid(True)

fig.suptitle(
    f'D1Q3 LBM：音圧場の時間発展  '
    f'(N={N_LBM}, L={L_TUBE*100:.0f}cm, τ={TAU_LBM}, c={C}m/s)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fd_lbm_wave_propagation.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()
print('右端（口唇端）で反射（位相反転）が起きていることに注目')

In [ ]:
# ---- LBM インパルス応答の計算 ----
DURATION_IR = 0.05   # [s]

print('LBM インパルス応答を計算中...')
lbm.reset()
ir_lbm_raw = lbm.run_impulse(DURATION_IR)

# 44100 Hz にリサンプル（LBM は高いサンプリングレートで動作）
n_target = int(DURATION_IR * FS)
ir_lbm   = signal.resample(ir_lbm_raw, n_target)
print(f'  LBM: {len(ir_lbm_raw)} ステップ → {n_target} サンプル @ {FS}Hz')

# ---- DWM インパルス応答（Kelly-Lochbaum 均一管）----
# N=44 セクション × OVERSAMPLE=2 → 内部 88200 Hz，出力 44100 Hz
# L_eff = N × c / (FS×2) = 44 × 343/88200 ≈ 17.1 cm ≈ L_TUBE
N_DWM     = 44
OVERSAMPLE = 2
print(f'DWM インパルス応答を計算中 (N={N_DWM}, oversample={OVERSAMPLE})...')
L_eff_dwm = N_DWM * C / (FS * OVERSAMPLE)
print(f'  DWM 有効管長 = {L_eff_dwm*100:.1f} cm  （LBM と比較：{L_TUBE*100:.0f} cm）')

r_dwm = np.zeros(N_DWM)
l_dwm = np.zeros(N_DWM)
ir_dwm = np.zeros(n_target)
step_total = 0

for n in range(n_target):
    for os in range(OVERSAMPLE):
        glot_in = 0.01 if step_total == 0 else 0.0
        step_total += 1
        new_r = np.empty(N_DWM)
        new_l = np.empty(N_DWM)
        new_r[0]  = glot_in + 0.75 * l_dwm[0]   # グロッタル端（準閉端）
        new_r[1:] = r_dwm[:-1]                   # 均一管：透過のみ
        new_l[:-1]= l_dwm[1:]
        ir_dwm[n] = r_dwm[-1]                    # 口唇端からの出力
        new_l[-1] = -0.85 * r_dwm[-1]            # 口唇端（準開放端）
        r_dwm, l_dwm = new_r, new_l

# ---- 周波数応答の比較 ----
N_FFT = 8192
f_lbm, H_lbm = signal.freqz(ir_lbm[:N_FFT], worN=N_FFT, fs=FS)
f_dwm, H_dwm = signal.freqz(ir_dwm[:N_FFT], worN=N_FFT, fs=FS)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(f_dwm, 20*np.log10(np.abs(H_dwm)+1e-12),
        color='#60a5fa', lw=2.0,
        label=f'DWM (N={N_DWM}, ×{OVERSAMPLE})', alpha=0.9)
ax.plot(f_lbm, 20*np.log10(np.abs(H_lbm)+1e-12),
        color='#f59e0b', lw=1.8, linestyle='--',
        label=f'LBM D1Q3 (N={N_LBM}, τ={TAU_LBM})', alpha=0.9)

# 理論共鳴周波数の縦線
for k in range(1, 7):
    fn = (2*k-1) * C / (4 * L_TUBE)
    if fn < 5000:
        ax.axvline(fn, color='#94a3b8', lw=0.8, ls=':',
                   label=f'理論 f{k}={fn:.0f}Hz' if k <= 3 else None)

ax.set_xlabel('周波数 [Hz]', fontsize=11)
ax.set_ylabel('振幅 [dB]', fontsize=11)
ax.set_title(f'均一管の周波数応答比較\n'
             f'(L≈{L_TUBE*100:.0f}cm, 閉端-開放端)', fontsize=10)
ax.set_xlim(0, 5000)
ax.legend(fontsize=9)
ax.grid(True)

ax2 = axes[1]
t_ms = np.arange(int(0.025*FS)) / FS * 1000
pk_d = np.max(np.abs(ir_dwm)) + 1e-10
pk_l = np.max(np.abs(ir_lbm)) + 1e-10
ax2.plot(t_ms, ir_dwm[:len(t_ms)] / pk_d,
         color='#60a5fa', lw=1.5, label='DWM', alpha=0.9)
ax2.plot(t_ms, ir_lbm[:len(t_ms)] / pk_l,
         color='#f59e0b', lw=1.3, linestyle='--', label='LBM', alpha=0.9)
ax2.set_xlabel('時間 [ms]', fontsize=11)
ax2.set_ylabel('正規化振幅', fontsize=11)
ax2.set_title('インパルス応答（先頭 25 ms）', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True)

fig.suptitle('D1Q3 LBM vs デジタル波動管（DWM）：均一管の比較', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fd_lbm_vs_dwm_comparison.png',
            dpi=120, bbox_inches='tight', facecolor='#1e293b')
plt.show()

# ---- 試聴（管共鳴の残響を聴く）----
def make_audio(ir, n_samples):
    x = ir[:n_samples].copy()
    pk = np.max(np.abs(x))
    return (x / pk * 0.8).astype(np.float32) if pk > 1e-10 else x

n_aud = int(DURATION_IR * FS)
print('\n▶ LBM インパルス応答（管共鳴の残響）:')
display(Audio(make_audio(ir_lbm, n_aud), rate=FS))
print('\n▶ DWM インパルス応答（比較）:')
display(Audio(make_audio(ir_dwm, n_aud), rate=FS))

---
## まとめ

### 各セクションで得られた知見

| セクション | 主な知見 |
|---|---|
| 1. Reynolds 数 | /s/ の収縮部では Re >> Re_crit（乱流遷移）。母音は層流支配 |
| 2. Lighthill | 双極音源は速度に U⁶ で比例。摩擦音が息の強さに非常に敏感な理由 |
| 3. 吸気ノイズ | OQ パラメータを変えるだけで 4 種類の声質が生成可能 |
| 4. Coanda 効果 | 確率的ジェット偏向で自然な jitter/shimmer が再現できる |
| 5. LBM | DWM と同等の共鳴構造を再現。2D/3D + 乱流への拡張が本命 |

### このプロジェクトへの次のステップ

#### 短期（実装コスト低）
- **吸気ノイズの統合**：`frication-model-phase3.ipynb` の波動管に OQ-dependent ノイズを追加し、ブレシー声・常態声・緊迫声を一貫したパラメータで制御する
- **Coanda モデルの採用**：`glottal-source-models.ipynb` の LF モデルに確率的偏向を組み込む

#### 中期（研究価値高）
- **準1次元流れ + DWM 連成**：声道断面積関数から Re・乱流強度を動的に計算し、DWM の音源として使う
- **LBM 声門局所モデル**：D1Q2/D1Q3 LBM で声門周辺（5mm 程度）の流れをシミュレートし、体積速度時系列を生成する

#### 長期（参照資料）
- Zheng ら（JASA 2011）の DNS+IB/FEM 連成を高忠実度ベンチマークとして参照
- OpenFOAM を用いた LES 声道シミュレーション（検証用）

### 参考文献

- Lighthill, M.J. (1952). On sound generated aerodynamically. *Proc. Roy. Soc. A*, 211, 564-587.
- Neubauer, J. et al. (2007). Coherent structures of the near field flow in a self-oscillating model of the vocal folds. *JASA*, 121(2), 1102-1118.
- Zheng, X. et al. (2011). Direct-numerical simulation of the glottal jet and vocal-fold dynamics. *JASA*, 130(1), 404-415.
- Birkholz, P. (2006). Noise source models for fricative consonants. *Proc. Tonal Aspects of Languages*.
- Freund, J.B. et al. (2009). On the application of the lattice Boltzmann method to glottal flow. *JASA*, 125(1), EL29-EL34.

詳細は `docs/notes/fluid-dynamics-and-sound-synthesis.md` を参照してください。